# 🏦 Credit Scoring Model — Exploratory Data Analysis

**CodeAlpha ML Internship | German Credit Dataset (UCI)**

This notebook covers:
1. Data loading & initial exploration
2. Data cleaning & quality checks
3. Exploratory Data Analysis (EDA)
4. Feature engineering
5. Model training & evaluation
6. Feature importance & SHAP analysis

In [ ]:
# ── Install dependencies if needed ────────────────────────────
# !pip install -r ../requirements.txt

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from utils.data_loader import load_and_clean_data, engineer_features, get_train_test_split
from utils.model_trainer import train_all_models, get_best_model, evaluate_model

# Seaborn dark theme
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ Libraries loaded!')

## 1. Data Loading

In [ ]:
# Load and clean the German Credit Dataset
df = load_and_clean_data('../data/german.data')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nTarget distribution:')
print(df['credit_risk'].value_counts())
df.head()

## 2. Data Quality

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values:\n', missing[missing > 0] if missing.sum() > 0 else 'None — dataset is complete!')

# Duplicates
print(f'\nDuplicate rows: {df.duplicated().sum()}')

# Basic stats
df.describe().T.round(2)

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['credit_risk'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#43D9AD', '#FF6B6B'],
    edgecolor='white', width=0.5
)
axes[0].set_title('Credit Risk Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Credit Risk (0=Default, 1=Creditworthy)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Default', 'Creditworthy'], rotation=0)

# Age distribution by risk
df_vis = df.copy()
df_vis['Status'] = df_vis['credit_risk'].map({1: 'Creditworthy', 0: 'Default'})
df_vis.groupby('Status')['age'].plot.hist(
    bins=20, alpha=0.7, ax=axes[1],
    color={'Creditworthy': '#43D9AD', 'Default': '#FF6B6B'}
)
axes[1].set_title('Age Distribution by Credit Risk', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age (years)')
axes[1].legend(['Creditworthy', 'Default'])

plt.tight_layout()
plt.savefig('../screenshots/eda_basic.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../screenshots/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Credit amount vs duration — colored by risk
fig = px.scatter(
    df_vis, x='duration', y='credit_amount',
    color='Status', opacity=0.6,
    color_discrete_map={'Creditworthy': '#43D9AD', 'Default': '#FF6B6B'},
    title='<b>Loan Duration vs Credit Amount</b>',
    labels={'duration': 'Duration (months)', 'credit_amount': 'Credit Amount (DM)'},
    template='plotly_dark',
    marginal_x='histogram', marginal_y='violin',
    width=900, height=550
)
fig.show()

## 4. Feature Engineering

In [ ]:
# Engineer features
df_eng, encoders = engineer_features(df)
print(f'Original features: {df.shape[1]}')
print(f'After engineering: {df_eng.shape[1]}')
print(f'\nNew derived features: debt_duration_ratio, installment_burden, age_group, credit_tier')
df_eng.head()

## 5. Model Training & Evaluation

In [ ]:
# Split data
X_train, X_test, y_train, y_test, feature_names, scaler = get_train_test_split(df_eng)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

# Train all models
trained_models, results = train_all_models(X_train, y_train, X_test, y_test)

# Summary table
summary = pd.DataFrame({
    name: {
        'Accuracy':  round(res['accuracy'], 4),
        'Precision': round(res['precision'], 4),
        'Recall':    round(res['recall'], 4),
        'F1 Score':  round(res['f1_score'], 4),
        'ROC-AUC':   round(res['roc_auc'], 4),
    } for name, res in results.items()
}).T

best = get_best_model(results)
print(f'\n🏆 Best Model: {best}')
summary

In [ ]:
# Feature importance for best tree model
from utils.visualizations import plot_feature_importance

best_model = trained_models[best]
fig = plot_feature_importance(best_model, feature_names, best, top_n=15)
fig.show()

In [ ]:
# SHAP Analysis (requires: pip install shap)
try:
    import shap
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test[:100])
    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values
    
    shap.summary_plot(sv, X_test[:100], feature_names=feature_names, plot_type='bar', show=True)
except ImportError:
    print('SHAP not installed. Run: pip install shap')
except Exception as e:
    print(f'SHAP error: {e}')

## 6. Save Final Model

In [ ]:
from utils.model_trainer import save_model_bundle

save_model_bundle(
    model=best_model,
    scaler=scaler,
    feature_names=feature_names,
    label_encoders=encoders,
    model_name=best,
    save_dir='../models'
)
print('✅ Model saved!')